In [ ]:
import numpy as np
from scipy.sparse import kron, eye, csr_matrix
from scipy.sparse.linalg import eigs

In [ ]:
W_A = 25.338776456203686  # storage-mode frequency
KAPPA_B = 5 / 10.4  # buffer single-photon loss rate
E_J = 37 * 2 * np.pi  # Josephson energy
PHI_A = 0.11  # zero-point phase fluctuation, storage mode
PHI_B = 0.204  # zero-point phase fluctuation, buffer mode
n_a = 25
n_b = 11
epsilon_p = .1
alpha_sq = 3
g = np.sin(epsilon_p) * E_J * PHI_A**2 * PHI_B
kappa_2 = 4 * g**2 / KAPPA_B
kappa_a = 0.005 * kappa_2
epsilon_d = 2 * alpha_sq * g

In [ ]:
def destroy(n):
    """Truncated bosonic annihilation operator on a Fock space of dim n."""
    return np.diag(np.sqrt(np.arange(1, n)), 1)


def build_liouvillian(H, jump_ops, N):
    """Vectorized Liouvillian: L such that d(vec rho)/dt = L @ vec(rho)."""
    Hs = csr_matrix(H)
    I = eye(N, format='csr')
    L = -1j * (kron(I, Hs) - kron(Hs.T, I))
    for Lk in jump_ops:
        Lk = csr_matrix(Lk)
        Lkd = Lk.conj().T
        LdL = Lkd @ Lk
        L += kron(Lk.conj(), Lk)
        L -= 0.5 * (kron(I, LdL) + kron(LdL.T, I))
    return L

In [ ]:
N = n_a * n_b

a = destroy(n_a)
b = destroy(n_b)
Ia, Ib = np.eye(n_a), np.eye(n_b)
a_tot = np.kron(a, Ib)
b_tot = np.kron(Ia, b)
non_linear_op = np.kron(a @ a, b.conj().T) + np.kron((a @ a).conj().T, b)
drive_op = b_tot + b_tot.conj().T
n_op = np.kron(a.conj().T @ a, Ib)

# a -> -a photon-number parity, tensored with identity on the buffer mode.
# This commutes with the full Liouvillian (Hamiltonian moves n_a by +-2;
# every jump operator's dissipator is parity-even even though a itself is
# parity-odd, since the two sign flips in D[a] cancel), so eigenvectors
# split exactly into even ("steady-state-like") and odd ("bit-flip-like").
parity_a = np.diag((-1.0) ** np.arange(n_a))
Parity = np.kron(parity_a, Ib)

def get_bit_flip(Parity, N, n_op, H,jump_ops):
    L = build_liouvillian(H, jump_ops, N)
    eigvals, eigvecs = eigs(L, k=6, sigma=0, which='LM')
    idx_sort = np.argsort(-eigvals.real)

    ss_val = bitflip_val = nbar = None
    for j in idx_sort:
        vec = eigvecs[:, j]
        rho = vec.reshape(N, N).T           # unvectorize
        rho_p = Parity @ rho @ Parity
        is_odd = np.linalg.norm(rho_p - rho) > np.linalg.norm(rho_p + rho)
        ev = eigvals[j]
        if abs(ev.real) < 1e-8 and ss_val is None:
            ss_val = ev
            rho_ss = rho / np.trace(rho)
            nbar = np.real(np.trace(rho_ss @ n_op))
        elif is_odd and bitflip_val is None:
            bitflip_val = ev

    if bitflip_val is None: return (nbar, np.nan)
    else : return (nbar, -bitflip_val.real)

In [ ]:
jump_ops = [
    np.sqrt(kappa_a) * a_tot,
    np.sqrt(KAPPA_B) * b_tot,
]

H = epsilon_d / 2 * drive_op + g * non_linear_op

L = build_liouvillian(H, jump_ops, N)
L

In [ ]:
eigvals, eigvecs = eigs(L, k=6, sigma=0, which='LM')

## Krylov–Schur on the plain RWA model (GPU cluster), one point, vs. the one-mode model

The shift-invert `eigs` above needs a sparse LU of the full `L` (75 625², the LU fill-in does not fit in RAM). `KrylovSchurLindblad` never builds `L`: it only propagates density matrices with `dq.mesolve`, with the same settings as the ATS pipeline (`floquet_lindblad.pipeline.solve_point_ks`).

- **RWA model** (time independent): `H = epsilon_d/2 (b + b†) + g (a² b† + a†² b)`, jumps `[sqrt(kappa_1) a, sqrt(kappa_b) b]`. `g`, `kappa_1`, `kappa_b`, `epsilon_d` come from `floquet_lindblad.models.ats._derived_params`, the same as for the ATS builders.
- **`T_block`** is a free time step here, since there's no drive period. The default is one ATS drive period `2π/w_a`. The rate is `-log(mu) / T_block` and must not depend on `T_block` (a cheap check: rerun with e.g. `T_block=10.0`).
- **One-mode reference**: the buffer adiabatically eliminated, `H = 0`, jumps `[sqrt(kappa_2) (a² + epsilon_d/(2g)), sqrt(kappa_1) a]` with `kappa_2 = 4 g²/kappa_b`. `L` is only `n_a² = 625` wide, so scipy `eigs` (shift-invert at 0) runs on the client in well under a second. The parameters are taken from the RWA result (`params`), so nothing is redefined. The two should agree up to the elimination's corrections, of order `(g/kappa_b)²`.

This section does not depend on the scipy cells above. Run it from the next cell down, in a **fresh kernel** (the bootstrap must set `RAY_AUTH_MODE` before `ray` is imported).

In [ ]:
# Cluster bootstrap, same as notebooks/solvers/krylov_schur.ipynb.
import os
import subprocess
import sys

assert sys.version_info[:2] == (3, 13), (
    f"kernel runs Python {sys.version.split()[0]}; the cluster needs 3.13.x. "
    "Select the workspace pixi environment as the kernel."
)

# Ray reads RAY_AUTH_MODE once, at its first import, and caches it. Setting it
# here works *only* because nothing has imported ray yet in a fresh kernel. If
# you have already run anything that imports ray, restart the kernel first.
os.environ["RAY_AUTH_MODE"] = "token"
os.environ.setdefault("VAULT_ADDR", "https://vault.int.alice-bob.com")

# Mint the cluster token from Vault here, so the notebook also works from a
# kernel that was not launched from a shell with RAY_AUTH_TOKEN exported (which
# is the normal case in VS Code). `vault login` is interactive, so it stays a
# manual, once-a-week step.
if not os.environ.get("RAY_AUTH_TOKEN"):
    try:
        os.environ["RAY_AUTH_TOKEN"] = subprocess.run(
            ["vault", "kv", "get", "-field=token", "kv-it/prod/ray/auth-token"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError) as exc:
        raise RuntimeError(
            "Could not mint the cluster token from Vault. In a terminal, run:\n"
            "  export VAULT_ADDR=https://vault.int.alice-bob.com\n"
            "  vault login -method=oidc role=research_core_user\n"
            "then re-run this cell (no kernel restart needed).\n"
            f"{(getattr(exc, 'stderr', '') or '').strip()}"
        ) from exc

import ray

from anb_compute import ray as acr

print("python", sys.version.split()[0], "| ray", ray.__version__)

import pathlib


def _find_repo():
    """Locate the repo checkout without hardcoding an absolute path.

    Notebooks have no __file__, so search upward from the working directory --
    which VS Code sets to either the notebook's folder or the workspace root,
    depending on `jupyter.notebookFileRoot`. Both are handled.
    """
    here = pathlib.Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / "src" / "floquet_lindblad").is_dir():
            return base
        for hit in sorted(base.glob("*/src/floquet_lindblad")):
            return hit.parents[1]
    raise RuntimeError(f"no checkout with src/floquet_lindblad found above {here}")


REPO = _find_repo()

# Shipped by PATH, not by name: a bare module name that resolves to a
# non-editable site-packages install ships a frozen copy, and local edits would
# never reach the worker. These prepend to the worker's sys.path, so the
# dynamiqs here shadows the image's stock 0.3.6 (which lacks `mesolve_fast`).
PKG = str(REPO / "src" / "floquet_lindblad")
# The patched dynamiqs checkout, expected beside the repo; $DYNAMIQS_SRC wins.
DYNAMIQS = os.environ.get("DYNAMIQS_SRC") or str(
    REPO.parent / "dynamiqs" / "dynamiqs"
)
for _p in (PKG, DYNAMIQS):
    if not pathlib.Path(_p).is_dir():
        raise RuntimeError(f"not a directory, cannot ship to workers: {_p}")

RUNTIME_ENV = acr.build_runtime_env(py_modules=[PKG, DYNAMIQS])

In [ ]:
def run_krylov_schur_rwa(
    *,
    alpha_sq,
    eps_p,
    n_a,
    n_b,
    T_block=None,
    # Krylov-Schur (defaults: those of `solve_point_ks`)
    m=50,
    k=12,
    res_tol=1e-10,
    max_cycles=35,
    rtol_loop=1e-9,
    atol_loop=1e-10,
    res_tol_loop=1e-8,
    max_cycles_loop=None,
    rtol_final=1e-11,
    atol_final=1e-12,
    n_blocks_final=50,
    seed=0,
    k_top=3,
):
    """`solve_point_ks`, on the plain RWA model instead of the full ATS, on one GPU worker.

    Same two tolerance phases, same final Rayleigh quotient on `n_blocks_final`
    blocks, and the same result keys (`rate_bf`, `res_rel_n`, `tail_a`, ...), so
    the two results can be compared key by key. The model is time independent,
    so there is no frame transformation and no `output_phase`. `T_block` is a
    free time step (default: one ATS drive period, 2 pi / w_a).

    Extra key: `odd_fraction` = ||rho - P rho P|| / (2 ||rho||) of the target,
    with P the storage-mode parity: 1 for a bit-flip (parity-odd) mode, 0 for
    an even one.

    Physics constants come from `floquet_lindblad.models.ats` (single source of truth).
    """
    import time

    import jax
    import jax.numpy as jnp
    import numpy as np

    import dynamiqs as dq
    from floquet_lindblad import KrylovSchurLindblad
    from floquet_lindblad.models.ats import (
        E_J,
        KAPPA_B,
        PHI_A,
        PHI_B,
        W_A,
        _derived_params,
        _two_mode_operators,
    )

    def py(v):
        """0-d jax/numpy scalar -> plain python float/complex."""
        a = np.asarray(v)
        return complex(a) if np.iscomplexobj(a) else float(a)

    lines = []

    def log(s):
        print(s)
        lines.append(s)

    t0 = time.time()
    dims = (n_a, n_b)

    # --- plain RWA model: time independent ---------------------------------
    params, T_drive = _derived_params(
        alpha_sq, W_A, KAPPA_B, E_J, PHI_A, PHI_B, eps_p, n_periods=1
    )
    T_block = float(T_drive) if T_block is None else float(T_block)
    g, epsilon_d = params["g"], params["epsilon_d"]

    a, b = _two_mode_operators(n_a, n_b)
    H = epsilon_d / 2 * (b + dq.dag(b)) + g * (
        a @ a @ dq.dag(b) + dq.dag(a) @ dq.dag(a) @ b
    )
    
    #RWA correction coming from sympy_rwa_ats.ipynb
    Ka  = g**2 / W_A * (40 * PHI_A**2 + 27 * PHI_B**2) / (6 * PHI_B**2)
    Kb  = g**2 / W_A * 2 * PHI_B**2 * (58 * PHI_A**2 + 25 * PHI_B**2) / (15 * PHI_A**4)
    chi = g**2 / W_A * (135 * PHI_A**2 + 92 * PHI_B**2) / (15 * PHI_A**2)
    Da  = -g**2 / W_A * (100 * PHI_A**4 + 135 * PHI_A**2 * PHI_B**2 + 46 * PHI_B**4) / (15 * PHI_A**2 * PHI_B**2)
    Db  = -g**2 / W_A * (135 * PHI_A**4 + 324 * PHI_A**2 * PHI_B**2 + 100 * PHI_B**4) / (30 * PHI_A**4)
    eps2 = -epsilon_d * g / (8 * W_A)

    na, nb = dq.dag(a) @ a, dq.dag(b) @ b
    
    H = H + (
        #Db * nb
        + Da * na
        - Ka / 2 * dq.dag(a) @ dq.dag(a) @ a @ a
        - Kb / 2 * dq.dag(b) @ dq.dag(b) @ b @ b
        - chi * na @ nb
        + eps2 * (dq.dag(a) @ dq.dag(a) + a @ a)
    )
    

    jump_ops = [jnp.sqrt(params["kappa_1"]) * a, jnp.sqrt(params["kappa_b"]) * b]

    solver = KrylovSchurLindblad(
        H, jump_ops, T_block, dims=dims, rtol=rtol_loop, atol=atol_loop
    )
    x0 = solver.make_x0(seed=seed)
    t_build = time.time() - t0

    # --- phase 1: cycles at the loop tolerance -------------------------------
    same_tol = (rtol_loop, atol_loop) == (rtol_final, atol_final)
    if max_cycles_loop is None:
        max_cycles_loop = max(1, max_cycles // 2)
    t1 = time.time()
    out = solver.solve(
        x0,
        m=m,
        k=k,
        res_tol=res_tol if same_tol else res_tol_loop,
        max_cycles=max_cycles if same_tol else max_cycles_loop,
        log=log,
    )
    history = list(out["history"])
    n_cycles_loop = out["n_cycles"]
    t_loop = time.time() - t1

    # --- phase 2: polish at the final tolerance, resuming the kept subspace ---
    t2 = time.time()
    if not same_tol:
        log(f"-- switching integrator tolerance to rtol={rtol_final:g}, atol={atol_final:g}")
        solver.set_tolerance(rtol_final, atol_final)
        out = solver.solve(
            state=out["state"],
            m=m,
            k=k,
            res_tol=res_tol,
            max_cycles=max(1, max_cycles - n_cycles_loop),
            log=log,
        )
        history += list(out["history"])
    t_polish = time.time() - t2

    x_ritz = out["x_ritz"]
    gap = max(float(1 - complex(out["mu"]).real), 1e-14)
    log(f"-- final Rayleigh quotient on {n_blocks_final} block(s): "
        f"1 - mu = {gap:.2e} per block, 1 - mu**n ~ {n_blocks_final * gap:.2e} "
        f"({n_blocks_final * gap / rtol_final:.1e} x rtol_final)")

    # --- final numbers on the true propagator, tight tolerance ---------------
    t3 = time.time()
    res_1 = solver.residual_check(x_ritz, n_blocks=1, rtol=rtol_final, atol=atol_final)
    res_n = solver.residual_check(
        x_ritz, n_blocks=n_blocks_final, rtol=rtol_final, atol=atol_final
    )
    t_final = time.time() - t3

    # --- truncation check and parity of the target (Fock basis already) -----
    rho = np.asarray(solver.unvec(x_ritz))
    D = np.einsum("ij,ij->i", rho, rho.conj()).real.reshape(n_a, n_b)
    w_a = D.sum(axis=1) / D.sum()
    w_b = D.sum(axis=0) / D.sum()
    parity = np.kron((-1.0) ** np.arange(n_a), np.ones(n_b))
    rho_p = parity[:, None] * rho * parity[None, :]
    odd_fraction = float(np.linalg.norm(rho - rho_p) / (2 * np.linalg.norm(rho)))

    return {
        "model": "rwa",
        "alpha_sq": float(alpha_sq),
        "eps_p": float(eps_p),
        "n_a": int(n_a),
        "n_b": int(n_b),
        # eigenvalue from the Krylov iteration (one block) ...
        "mu": complex(out["mu"]),
        "rate_ritz": py(solver.rate_from_mu(out["mu"])),
        "res_rel": float(out["res_rel"]),
        "converged": bool(out["converged"]),
        # ... and from the final Rayleigh quotients at tight tolerance
        "mu_RQ_1": py(res_1["mu_RQ"]),
        "res_rel_1": py(res_1["res_rel"]),
        "mu_RQ_n": py(res_n["mu_RQ"]),
        "rate_bf": py(res_n["rate_RQ"]),  # named as in `solve_point_ks`
        "res_rel_n": py(res_n["res_rel"]),
        "n_blocks_final": int(n_blocks_final),
        "gap": gap,
        # cost
        "n_apply": int(out["n_apply"]),
        "n_cycles": len(history),
        "n_cycles_loop": int(n_cycles_loop),
        "res_hist": np.array([h["res_rel"] for h in history]),
        "apply_hist": np.array([h["n_apply"] for h in history]),
        "mu_hist": np.array([h["mu"] for h in history]),
        "log": lines,
        # truncation check, parity
        "fock_weight_a": w_a,
        "fock_weight_b": w_b,
        "tail_a": float(w_a[-k_top:].sum()),
        "tail_b": float(w_b[-k_top:].sum()),
        "k_top": int(k_top),
        "odd_fraction": odd_fraction,
        "params": {kk: py(v) for kk, v in params.items()},
        "T_block": T_block,
        "settings": {
            "n_a": n_a, "n_b": n_b, "m": m, "k": k, "T_block": T_block,
            "res_tol": res_tol, "res_tol_loop": res_tol_loop,
            "max_cycles": max_cycles, "max_cycles_loop": max_cycles_loop,
            "rtol_loop": rtol_loop, "atol_loop": atol_loop,
            "rtol_final": rtol_final, "atol_final": atol_final,
            "n_blocks_final": n_blocks_final, "seed": seed,
        },
        "timings_s": {
            "build": t_build,
            "loop": t_loop,
            "polish": t_polish,
            "final_RQ": t_final,
            "total": time.time() - t0,
        },
        "device": str(jax.devices()[0]),
    }



In [ ]:
import time

# One point. Krylov-Schur settings: the `solve_point_ks` defaults, written out.
POINT = dict(alpha_sq=3.0, eps_p=0.1, n_a=25, n_b=11)
KS = dict(
    m=50, k=12, res_tol=1e-10, max_cycles=35,
    rtol_loop=1e-9, atol_loop=1e-10, res_tol_loop=1e-8,
    rtol_final=1e-11, atol_final=1e-12, n_blocks_final=50,
    seed=0,
)
RUNS = {
    "RWA": {},  # T_block = 2 pi / w_a by default
}

results = {}
for name, extra in RUNS.items():
    t0 = time.time()
    results[name] = acr.run(
        run_krylov_schur_rwa,
        num_gpus=.5,
        num_cpus=4,
        runtime_env=RUNTIME_ENV,
        poll_seconds=5,
        **POINT, **KS, **extra,
    )
    r = results[name]
    print(f"\n[{name}] wall clock incl. startup: {time.time() - t0:.1f} s | device {r['device']}")
    print("  timings:", {kk: round(v, 1) for kk, v in r["timings_s"].items()})
    print("\n".join("  " + s for s in r["log"]))

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix, diags, eye, kron
from scipy.sparse.linalg import eigs


def liouvillian(H, jump_ops):
    """Column-major vectorized Lindbladian, d vec(rho)/dt = L @ vec(rho) (as `build_liouvillian` above)."""
    n = H.shape[0]
    I = eye(n, format="csr")
    H = csr_matrix(H)
    L = -1j * (kron(I, H) - kron(H.T, I))
    for J in map(csr_matrix, jump_ops):
        JdJ = J.conj().T @ J
        L += kron(J.conj(), J) - 0.5 * (kron(I, JdJ) + kron(JdJ.T, I))
    return L.tocsr()


def one_mode_bit_flip(n_a, g, epsilon_d, kappa_2, kappa_1, k=4):
    """Bit-flip rate of the adiabatically eliminated (one-mode) model, by shift-invert `eigs`.

    Eliminating b from H = g (a^2 + epsilon_d / (2g)) b^dag + h.c. (the RWA model
    above) gives the two-photon jump sqrt(kappa_2) (a^2 + epsilon_d / (2g)): the
    same cat, |alpha|^2 = alpha_sq, as the two-mode model. The parity
    rho -> P rho P is a symmetry, and the bit-flip mode is the slowest mode of the
    odd block. That block does not hold the steady state, so it is not singular
    and sigma = 0 is safe.
    """
    a = diags(np.sqrt(np.arange(1, n_a)), 1, format="csr", dtype=complex)
    jump_ops = [
        np.sqrt(kappa_2) * (a @ a + epsilon_d / (2 * g) * eye(n_a, format="csr")),
        np.sqrt(kappa_1) * a,
    ]
    L = liouvillian(csr_matrix((n_a, n_a), dtype=complex), jump_ops)

    p = (-1.0) ** np.arange(n_a)
    odd = np.flatnonzero(np.kron(p, p) < 0)  # rho[m, n] with n_m + n_n odd
    vals = eigs(L[odd][:, odd].tocsc(), k=k, sigma=0, which="LM", return_eigenvectors=False)
    lam = vals[np.argmax(vals.real)]
    return {"rate_bf": -lam.real, "lam": lam, "vals_odd": vals, "dim": n_a**2}


# Same point, same derived parameters as the RWA run.
p = results["RWA"]["params"]
one_mode_args = dict(g=p["g"], epsilon_d=p["epsilon_d"], kappa_2=p["kappa_2"], kappa_1=p["kappa_1"])
one_mode = one_mode_bit_flip(POINT["n_a"], **one_mode_args)

# Truncation check of the reference itself (cheap: L is n_a^2 wide).
print(f"{'n_a':>4} {'rate_bf (one mode)':>19}")
for n in (15, 20, 25, 30, 35):
    print(f"{n:4d} {one_mode_bit_flip(n, **one_mode_args)['rate_bf']:19.6e}")

In [ ]:
import numpy as np

print(f"{'model':<6} {'rate_bf (RQ)':>14} {'rate (Ritz)':>13} {'res Krylov':>11} "
      f"{'res n blk':>10} {'conv':>5} {'apps':>5} {'cycles':>6} {'tail_a':>9} {'tail_b':>9} "
      f"{'odd':>5} {'GPU s':>7}")
for name, r in results.items():
    print(
        f"{name:<6} {complex(r['rate_bf']).real:14.6e} {complex(r['rate_ritz']).real:13.6e} "
        f"{r['res_rel']:11.2e} {abs(r['res_rel_n']):10.2e} {str(r['converged']):>5} "
        f"{r['n_apply']:5d} {r['n_cycles']:6d} {r['tail_a']:9.1e} {r['tail_b']:9.1e} "
        f"{r['odd_fraction']:5.2f} "
        f"{r['timings_s']['loop'] + r['timings_s']['polish']:7.1f}"
    )
    print(f"[{name}] T_block = {r['T_block']:.6g}, 1 - Re mu = {r['gap']:.3e} per block, "
          f"Im mu = {complex(r['mu']).imag:.1e} (noise indicator)")

rate_1m = one_mode["rate_bf"]
print(f"\none mode (scipy eigs, n_a = {POINT['n_a']}, dim L = {one_mode['dim']}): "
      f"rate_bf = {rate_1m:.6e}")
for name, r in results.items():
    rate = complex(r["rate_bf"]).real
    print(f"{name} / one mode = {rate / rate_1m:.4f}   ({name} - one mode) / one mode = "
          f"{rate / rate_1m - 1:+.2%}")